# Task 2: Predicting Rainfall
## Singapore Rainfall, 2017–2024

## 2.1 Spark Session

In [1]:
import socket
from pyspark.sql import SparkSession
from pyspark.sql import Window

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyspark.sql.functions as F
import seaborn as sns
import pandas as pd

import os
import sys

In [2]:
spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")

print("Running on:   ", socket.gethostname())
print("Spark master: ", spark.sparkContext.master)
print("Spark version:", spark.version)

Running on:    rainfall-dev-m
Spark master:  yarn
Spark version: 3.5.3


In [3]:
# Constants for Central Coordinates and Rainfall Multiplier per Day
# There are 12 five-minute intervals per hour and 24 hours per day
C_LAT, C_LON = 1.3521, 103.8198 
RF_MULTIPLIER = (24 * 60) / 5

## 2.2 Reading the Cleaned Rainfall Data

In [4]:
rain = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/parquet/")

## 2.3 Seasonal Patterns: Which Months Are Wetter or Drier?

This section examines the typical monthly rainfall pattern across 2017–2024. Estimated rainfall is first calculated for each year–month, then averaged across years to produce a January–December profile.

The resulting chart shows which calendar months tend to be wetter or drier during the study period. These estimates are based on available observations and may be affected by missing records and changes in the reporting stations.

In [ ]:
# Step 1: Extract the year and month from the observation date
rain = (
    rain.withColumn("date", F.to_date("date"))
        .withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
)

# Step 2: Calculate mean rainfall per observed five-minute interval
# for each year and month
monthly_spk = (
    rain.filter(F.col("year").between(2017, 2024))
        .groupBy("year", "month")
        .agg(
            F.avg("reading_value").alias("mean_per_interval"),
            F.countDistinct("station_id").alias("n_stations")
        )
        .withColumn(
            "year_month",
            F.make_date("year", "month", F.lit(1))
        )
        .withColumn(
            "days",
            F.dayofmonth(F.last_day("year_month"))
        )
        # mean interval rainfall × expected intervals in the month
        .withColumn(
            "avg_rainfall_monthly",
            F.col("mean_per_interval") * F.col("days") * RF_MULTIPLIER
        )
        .orderBy("year_month")
)

## 1.8 Auto-Correlation of Rainfall and Its Timelags 

- We also examine how strongly rainfall at a station relates to its own recent past. The autocorrelation at lag $k$ measures the correlation between $X_t$ and $X_{t-k}$, computed here for lags of five minutes (k = 1) to a day (k = 288). This tells us how far back readings remain informative, which determines how many lagged features are worth building for the prediction task.

In [6]:
def lookback(seconds):
    return (Window.partitionBy("station_id")
            .orderBy("ts_long")
            .rangeBetween(-seconds, 0))

In [7]:
rain_lag = rain.withColumn("ts_long", F.col("timestamp").cast("long"))

w = Window.partitionBy("station_id").orderBy("ts_long")

lags = {"lag_5m": 1, "lag_30m": 6, "lag_1h": 12}

for name, k in lags.items():
    rain_lag = (rain_lag
        .withColumn(name, F.lag("reading_value", k).over(w))
        .withColumn(name + "_ts", F.lag("ts_long", k).over(w)))
    
    # null out where the gap isn't the expected duration
    rain_lag = rain_lag.withColumn(name,
        F.when(F.col("ts_long") - F.col(name + "_ts") == k * 300,
               F.col(name)))
    rain_lag = rain_lag.drop(name + "_ts")

In [ ]:
future_30m = (Window.partitionBy("station_id")
                .orderBy("ts_long")
                .rangeBetween(1, 1800))

rain_lag = (rain_lag
    # rolling sums
    .withColumn("rain_cumsum_30m", F.sum("reading_value").over(lookback(1800)))
    .withColumn("rain_cumsum_1h",  F.sum("reading_value").over(lookback(3600)))
    # time features
    .withColumn("hour", F.hour("timestamp"))
    # target
    .withColumn("future_30m", F.sum("reading_value").over(future_30m)))

In [9]:
feature_cols = [
    "location_latitude", "location_longitude",
    "hour", "month",
    "reading_value",
    "lag_5m", "lag_30m", "lag_1h",
    "rain_cumsum_30m", "rain_cumsum_1h",
]

THRESHOLD = 1.0

feature_df = rain_lag.withColumn("label", (F.col("future_30m") > THRESHOLD).cast("int"))

train = feature_df.filter(F.col("year") <= 2022)
val   = feature_df.filter(F.col("year") == 2023)
test  = feature_df.filter(F.col("year") == 2024)

train.select(F.avg("label").alias("positive_rate")).show()

+--------------------+
|       positive_rate|
+--------------------+
|0.023092723164791808|
+--------------------+

